# M2.1.1 verified-source recovery

Compatibility launch surface; uses the same canonical package and readiness launcher as `m2_colab.ipynb`.

Pin the exact reviewed SHA. The cells contain no independent acquisition, QC, alignment, liftover, or domain science. Gate B remains unresolved. Code verification and cache recovery cannot create canonical completion. Expected source cache is about 4.9 GB; hydration needs a separate local copy plus headroom. Never use Drive as Nextflow work/.


In [ ]:
from pathlib import Path
import os
import re
import subprocess
import sys

REPOSITORY_URL = "https://github.com/jcollins-bioinfo/giab-wes-nextflow.git"
REPOSITORY_REF = ""  # REQUIRED: reviewed 40-character commit SHA from the handoff
REPOSITORY_DIR = Path("/content/giab-wes-nextflow")
if not re.fullmatch(r"[0-9a-f]{40}", REPOSITORY_REF):
    raise ValueError("Set the exact reviewed commit SHA before running this notebook.")
if not REPOSITORY_DIR.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
origin = subprocess.run(["git", "-C", str(REPOSITORY_DIR), "remote", "get-url", "origin"], check=True, text=True, capture_output=True).stdout.strip()
if origin != REPOSITORY_URL:
    raise ValueError("Unexpected repository origin; use a fresh clone of the named repository.")
status = subprocess.run(["git", "-C", str(REPOSITORY_DIR), "status", "--porcelain", "--untracked-files=normal"], check=True, text=True, capture_output=True).stdout
if status:
    raise ValueError("Existing checkout is modified. Preserve it and use a fresh repository directory.")
subprocess.run(["git", "-C", str(REPOSITORY_DIR), "fetch", "origin", REPOSITORY_REF], check=True)
RESOLVED_SHA = subprocess.run(["git", "-C", str(REPOSITORY_DIR), "rev-parse", "--verify", "FETCH_HEAD^{commit}"], check=True, text=True, capture_output=True).stdout.strip()
if RESOLVED_SHA != REPOSITORY_REF:
    raise ValueError("Fetched commit differs from the reviewed SHA.")
subprocess.run(["git", "-C", str(REPOSITORY_DIR), "checkout", "--detach", RESOLVED_SHA], check=True)
ENV = dict(os.environ, PYTHON=sys.executable, REPOSITORY_REF=RESOLVED_SHA)
subprocess.run(["bash", "scripts/run_m2_readiness.sh", "identity"], cwd=REPOSITORY_DIR, env=ENV, check=True)
subprocess.run([sys.executable, "-I", "-m", "pip", "install", str(REPOSITORY_DIR)], check=True)
subprocess.run([sys.executable, "-I", "-c", "import giab_wes_nextflow; print(giab_wes_nextflow.__version__)"], check=True)
subprocess.run([sys.executable, "-I", "-m", "giab_wes_nextflow.runtime_identity", "--source-root", str(REPOSITORY_DIR), "--expected-sha", RESOLVED_SHA], check=True)
subprocess.run(["bash", "scripts/run_m2_readiness.sh", "verify"], cwd=REPOSITORY_DIR, env=ENV, check=True)


## Optional owner-run private cache operation
Run only after selecting the intended operation and inspecting preflight. A cached source inventory is distinct from prepared references and canonical domains.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/giab-wes-nextflow-private")
STAGING = Path("/content/m2-stage")
RUN_ID = ""  # REQUIRED: deliberate reusable acquisition/mirror ID
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", RUN_ID):
    raise ValueError("Set a deliberate reusable RUN_ID.")
ENV.update(DRIVE_ROOT=str(DRIVE_ROOT), STAGING=str(STAGING), RUN_ID=RUN_ID)
subprocess.run(["bash", "scripts/run_m2_readiness.sh", "preflight"], cwd=REPOSITORY_DIR, env=ENV, check=True)


In [ ]:
# A code/preflight success does not authorize or establish canonical execution.
MODE = ""  # Choose hydrate, acquire, or mirror only after reviewing the preflight.
# For an older cache, use exact historical source-manifest bytes and a NEW recovery ID.
SOURCE_MANIFEST = ""
RECOVERY_RUN_ID = ""
ENV.pop("SOURCE_MANIFEST", None)
ENV.pop("RECOVERY_RUN_ID", None)
if SOURCE_MANIFEST:
    ENV["SOURCE_MANIFEST"] = SOURCE_MANIFEST
if RECOVERY_RUN_ID:
    ENV["RECOVERY_RUN_ID"] = RECOVERY_RUN_ID
if MODE not in {"hydrate", "acquire", "mirror"}:
    raise ValueError("Choose the intended source-cache operation explicitly.")
subprocess.run(["bash", "scripts/run_m2_readiness.sh", MODE], cwd=REPOSITORY_DIR, env=ENV, check=True)
print("Source-cache operation returned successfully. Gate B BLOCKED; no canonical completion is claimed.")
